## Reading 33-1 - Numerical Correctness: When the Bits Run Out

By the end of this reading, you will learn:
<ol>
    <li>What <i>machine epsilon</i> is, and how to measure it experimentally</li>
    <li>How a rounding error smaller than one ten-millionth of a second caused the Patriot missile failure</li>
    <li>Why errors that are harmless once become dangerous when they accumulate</li>
    <li>Why flight software keeps time as an integer count, not a floating-point number</li>
</ol>

### Getting the Reading Code

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your Reading Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir reading33
    cd reading33

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/readings/lec33/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    accumulate.c  blas_dgemm.c  float_epsilon.c  Makefile  patriot.c  row_vs_col.c

### A Fixed Number of Bits

At the start of the semester, we converted decimal values to IEEE 754 by hand. The key fact from that work is simple: a <code>float</code> has 32 bits and a <code>double</code> has 64. A finite number of bits can only represent a finite set of numbers, so almost every real number gets <i>rounded</i> to the nearest one that fits.

How big is that rounding? The standard way to measure it is <b>machine epsilon</b>: the gap between 1.0 and the next larger number the type can represent. Any amount smaller than about half of epsilon, added to 1.0, simply disappears.

### Measuring Epsilon - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec33/float_epsilon.c">float_epsilon.c</a>

We can find epsilon experimentally. Start at 1.0 and keep cutting a value in half until adding it to 1.0 no longer changes anything:

    volatile float eps = 1.0f;
    volatile float test = 2.0f;
    int halvings = 0;

    while (test != 1.0f) {
    	eps = eps / 2.0f;
    	test = 1.0f + eps;
    	halvings++;
    }

    /* The loop stops one step too far: eps is now too small to matter */
    float machine_eps = eps * 2.0f;

    $ make float_epsilon
    Halvings:           24
    Measured epsilon:   1.1920928955e-07
    FLT_EPSILON:        1.1920928955e-07

Our measurement matches <code>FLT_EPSILON</code> from <code>&lt;float.h&gt;</code> exactly, and the number of halvings matches what we know from IEEE 754: a <code>float</code> has 23 stored fraction bits, plus the hidden leading 1.

Why <code>volatile</code>? It tells the compiler that every assignment must actually be stored as a 32-bit <code>float</code>. Without it, the compiler is allowed to keep the values in wider registers, and the experiment would measure the register instead of the type. You will see why this matters in Homework 9, where you will measure epsilon for <code>double</code> and for the 80-bit <code>long double</code>.

### The Patriot Missile, Reconstructed - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec33/patriot.c">patriot.c</a>

In Reading 32-1, we saw that the Patriot battery at Dhahran missed an incoming missile in 1991. Now we can compute exactly why.

The system counted time in tenths of a second and converted that count to seconds by multiplying by 0.1. The value 0.1 was stored as a binary fraction with 23 bits after the binary point. Since 0.1 is a repeating fraction in binary, everything after bit 23 was chopped off:

<ul>
    <li>0.1 &times; 2<sup>23</sup> = 838,860.8, which was stored as the integer 838,860</li>
    <li>The stored value of "0.1" was really 0.099999904633</li>
</ul>

The program multiplies the tick count by that stored value, exactly as the Patriot did:

    $ make patriot
    Stored value of 0.1: 0.099999904633
    Error per tick:      0.000000095367 seconds

       Hours  True time (s)    Patriot (s)    Error (s)
           1      3600.0000      3599.9966       0.0034
           8     28800.0000     28799.9725       0.0275
          20     72000.0000     71999.9313       0.0687
          48    172800.0000    172799.8352       0.1648
          72    259200.0000    259199.7528       0.2472
         100    360000.0000    359999.6567       0.3433

The error per tick is less than one ten-millionth of a second. But the clock ticked ten times per second, and the battery had been running for about 100 hours. By then the error was 0.34 seconds. An incoming Scud travels roughly 1,700 meters per second, so the system looked for the target more than half a kilometer from where it actually was. The missile struck a barracks, killing 28 soldiers. Corrected software arrived the following day.

Notice that the code does nothing "wrong" in the sense of Reading 32-2: there is no undefined behavior, no overflow, and no warning. Every tool we have would pass it.

### Errors That Accumulate - <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec33/accumulate.c">accumulate.c</a>

The Patriot error grew because it was <i>multiplied</i> by a large tick count. Adding a small value over and over causes a similar problem. This program keeps track of one day of 0.1-second ticks three different ways:

    for (uint32_t i = 0; i < TICKS_PER_DAY; i++) {
    	time_float += 0.1f;
    	time_double += 0.1;
    	ticks++;
    }

    $ make accumulate
    Expected:            86400.000000 s
    float accumulator:   87145.843750 s
    double accumulator:  86400.000001 s
    integer tick count:  864000 ticks = 86400.0 s

After one day, the <code>float</code> clock is off by <b>more than 12 minutes</b>. Here is why: as the total grows, the gap between representable <code>float</code> values grows too. Near 86,400, neighboring <code>float</code> values are about 0.008 apart, so every addition of 0.1 is rounded, and those roundings do not cancel out.

The <code>double</code> clock is off by only a microsecond after one day, but it still drifts, and a spacecraft runs for years. The integer count is <i>exact</i>, forever, until it overflows (which is a different problem, with a different fix, that we will cover later).

This is why the first rules of embedded timekeeping are to store time as integers and convert to seconds only when you need to display it. It is also the difference between <b>precision</b> and <b>accuracy</b>: the <code>float</code> clock prints six decimal places of precision while being wrong by 745 seconds.

### <font color = "red">Class Introduction Question #1 - What is machine epsilon, and what happens when you add a value much smaller than epsilon to 1.0?</font>

### <font color = "red">Class Introduction Question #2 - The Patriot's error per tick was about 0.000000095 seconds. How did such a small error become large enough to miss a missile?</font>

### <font color = "red">Class Introduction Question #3 - Why does the <code>float</code> accumulator drift so much more than the <code>double</code> accumulator, and why does the integer tick count not drift at all?</font>

### The next reading for this lecture is <a href = "https://github.com/mmorri22/cse20133/blob/main/readings/lec33/Reading%2033-2.ipynb">Reading 33-2 - Performance Correctness: Cache, GEMM, and BLAS</a>